In [1]:
#pip install jupyter pandas scikit-learn

In [2]:
import pandas as pd

df = pd.read_csv('../data/all_data.csv')
df_eng = df[df['Review_Language'] == 'English']
df_eng.to_csv('../data/data_eng.csv')

In [3]:
df = pd.read_csv('../data/data_eng.csv')
df.head()

,Unnamed: 0.1,Unnamed: 0,Review_Language,City Name,Hotel Name,Upside_Review,Downside_Review,Review_Score,Sentiment,source,Prompt_Language,na_up_review,na_down_review
0,985,0,English,Ankara,Hotel Ickale,Staff. Serkan türk was a great help during our...,NaN,10.0,POS,0,NaN,False,True
1,986,1,English,Ankara,Hotel Ickale,Nicely furnished room and nicely decorated lob...,NaN,9.0,POS,0,NaN,False,True
2,987,2,English,Ankara,Hotel Ickale,"Very central, near 2 metro stations. Room was ...",The wifi had annoying login that required logg...,9.0,POS,0,NaN,False,False
3,988,3,English,Ankara,Hotel Ickale,Lovely interior.. great location.. friendly staff,what is not to like\nrestaurant staff could us...,9.0,POS,0,NaN,False,False
4,989,4,English,Ankara,Hotel Ickale,"Thanks to Mr.Serkan, our problems were solved",NaN,9.0,POS,0,NaN,False,True


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/data_eng.csv')

# label: source column, check which value is fake
print(df['source'].value_counts())
print(df.groupby('source')['Prompt_Language'].apply(lambda s: s.notna().sum()))

# text: combine upside and downside review, missing values become empty strings
df['text'] = df['Upside_Review'].fillna('') + ' ' + df['Downside_Review'].fillna('')
df['text'] = df['text'].str.strip()

strata = df["source"].astype(str) + "_" + df["Sentiment"].astype(str)
# then pass stratify=strata

# stratified 75/25 split, whole rows kept so Sentiment stays aligned for question 3
train_df, test_df = train_test_split(
    df, test_size=0.25, stratify=strata, random_state=42
)

print(len(train_df), len(test_df))
print(train_df['source'].value_counts())
print(test_df['source'].value_counts())

train_df.to_csv('../data/train.csv', index=False)
test_df.to_csv('../data/test.csv', index=False)


source
0    1000
1    1000
Name: count, dtype: int64
source
0       0
1    1000
Name: Prompt_Language, dtype: int64
1500 500
source
1    750
0    750
Name: count, dtype: int64
source
1    250
0    250
Name: count, dtype: int64


## Bag of words and Matrix


In [5]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer



train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

# empty texts are read back from csv as NaN
train_df['text'] = train_df['text'].fillna('')
test_df['text'] = test_df['text'].fillna('')

y_train = train_df['source']
y_test = test_df['source']

# bag-of-words: lowercase, words in fewer than 5 training reviews dropped
vec = CountVectorizer(lowercase=True, min_df=5, token_pattern=r'(?u)\b[^\W\d_]{2,}\b')

# vocabulary learned on the training set only
X_train = vec.fit_transform(train_df['text'])

# test set uses the training vocabulary, no refitting
X_test = vec.transform(test_df['text'])


In [6]:
#Tinkering: Separating the test set into 1200 training and 300 testng reviews
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=0)

In [8]:
from sklearn.naive_bayes import MultinomialNB
from sklearn import tree

# Naive Bayes
NB = MultinomialNB()
NB.fit(X_tr, y_tr)

#Logistic regression

#Single tree

ST = tree.DecisionTreeClassifier()
ST = ST.fit(X_tr, y_tr)

#Random forests

#Gradient boosting

In [9]:
#logistic regression
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression()
logreg.fit(X_tr, y_tr)

y_pred = logreg.predict(X_val)


In [10]:
#Single tree
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
ST = DecisionTreeClassifier(criterion='gini')
ST = ST.fit(X_tr, y_tr)
tree_pred = ST.predict(X_val)
acc = metrics.accuracy_score(tree_pred, y_val)
print(acc)

0.7766666666666666


In [11]:
#Random forests
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

RF = RandomForestClassifier()
RF = RF.fit(X_tr, y_tr)
RF_pred = RF.predict(X_val)
print(metrics.accuracy_score(RF_pred, y_val))

0.87


In [15]:
from sklearn.ensemble import GradientBoostingClassifier

GB = GradientBoostingClassifier()
GB = GB.fit(X_tr, y_tr)
GB_pred = GB.predict(X_val)
print(metrics.accuracy_score(GB_pred, y_val))

0.8533333333333334


In [12]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(ST, X_train, y_train, cv=5)
scores

array([0.79666667, 0.78333333, 0.82      , 0.78333333, 0.82666667])

In [13]:
print(scores.mean(), scores.std())

0.8019999999999999 0.01820866704499688


# Cross validation vocabulary loop

In [ ]:
for k in folds:


# Cross validation and hyperparameter tuning